# Acceso digital y condiciones habitacionales: Censo 2024

**Pregunta:** ¿qué condiciones del hogar y de su vivienda aparecen asociadas
con disponer o no disponer de internet fija y computador, y cómo cambian sus
frecuencias entre regiones?

La unidad es un **hogar particular**, identificado por `(id_vivienda, id_hogar)`.
La unión con viviendas se hace por `id_vivienda`, muchos a uno. Se utilizan todos
los casos completos del país: no se recorta la base a unos miles de filas.

Fuentes: [portal oficial del INE](https://censo2024.ine.gob.cl/resultados/),
[manual y diccionario](https://censo2024.ine.gob.cl/wp-content/uploads/2025/12/manual_uso_microdatos_censo2024.pdf).
El manual documenta códigos, universos y llaves. El XLSX enlazado en el catálogo
respondió 404 durante la descarga; se conserva esa incidencia en el manifiesto.
Las bases CSV y el manual sí están disponibles y sus hashes se verifican.

Las proporciones son descriptivas del censo. No estiman pobreza, causas,
conectividad actual ni elegibilidad para beneficios.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments/protocolo_censo_2024.json').exists())
sys.path.insert(0,str(root / 'experiments'))
import censo_2024 as study
REGENERAR = False  # True vuelve a procesar las bases CSV oficiales completas.
if REGENERAR:
    study.main()
    import verificar_recuento_censo
    verificar_recuento_censo.main()
out = root / 'results/censo_2024'
model = json.loads((out / 'modelo.json').read_text())
audit = json.loads((out / 'calidad_datos.json').read_text())
protocol = json.loads(study.PROTOCOL.read_text())
counts = np.load(out / 'conteos_perfiles.npz')['counts']
for f in model['source']['files']:
    if f.get('available',True):
        assert hashlib.file_digest((root / f['file']).open('rb'),'sha256').hexdigest() == f['sha256']
assert hashlib.file_digest(study.PROTOCOL.open('rb'),'sha256').hexdigest() == model['protocol_sha256']
print('Fuentes originales y protocolo: hashes verificados.')
print('Descarga:',model['source']['retrieved_at'])
print('Archivos oficiales:',[(f['nombre'], f.get('bytes',f.get('http_status'))) for f in model['source']['files']])


Fuentes originales y protocolo: hashes verificados.
Descarga: 2026-10-05T07:47:42.349214+00:00
Archivos oficiales: [('Base de microdatos - Viviendas Censo 2024 (csv)', 55921195), ('Base de microdatos - Hogares Censo 2024 (csv)', 44678452), ('Manual de Uso de Microdatos Censo 2024', 2279393), ('Diccionario de variables microdatos Censo 2024', 404)]


## Variables y preparación

Se filtra `tipo_operativo == 2`. Las filas de colectivos y personas en situación
de calle representan otro operativo; no se cuentan como hogares particulares.
La fuente de agua y el índice de hacinamiento provienen de la vivienda y se
asignan a cada hogar que comparte esa vivienda. No se utilizan sus componentes
para minar reglas que reproduzcan la fórmula del hacinamiento.

Área, tenencia, tipología, agua y hacinamiento pueden integrar antecedentes;
computador e internet fija son los consecuentes. Se excluye mezclar categorías
de una misma variable, usar los identificadores como características o hacer
reglas entre las dos respuestas de acceso digital. Se conservan las categorías
originales del diccionario, con etiquetas abreviadas.


In [2]:
variables = pd.DataFrame([{'Variable':g[2], 'Nombre':g[1], 'Categorías':len(g[3]), 'Papel':'Antecedente' if j < 5 else 'Consecuente'} for j,g in enumerate(study.GROUPS)])
print(variables.to_string(index=False))
print('\nCategorías y códigos:')
print(pd.DataFrame(model['items'])[['source_variable','value','label']].to_string(index=False))


               Variable                      Nombre  Categorías       Papel
                   area                        Área           2 Antecedente
       p12_tenencia_viv                    Tenencia           9 Antecedente
        tipologia_hogar               Tipo de hogar           7 Antecedente
         p6_fuente_agua              Fuente de agua           4 Antecedente
    indice_hacinamiento Hacinamiento de la vivienda           3 Antecedente
        p15b_serv_compu                  Computador           2 Consecuente
p15d_serv_internet_fija               Internet fija           2 Consecuente

Categorías y códigos:
        source_variable  value                                        label
                   area      1                                       Urbana
                   area      2                                        Rural
       p12_tenencia_viv      1                                Propia pagada
       p12_tenencia_viv      2                             Propia

## Tamaño, llaves y datos faltantes

NA, -99 y cualquier código ajeno al diccionario se consideran no utilizables;
no se transforman en una respuesta «No». Se trabaja con casos completos en
las siete variables. Las exclusiones por variable se solapan y no deben sumarse.
Se comprueba la unicidad de la clave de hogar, la referencia a vivienda y la
coincidencia de región, área y operativo después de la unión.


In [3]:
for key in ['household_file_rows','excluded_other_operations','private_households','complete_households','excluded_incomplete','dwelling_file_rows','private_household_dwellings','duplicate_households','join_unmatched']:
    print(key,':',format(audit[key],','))
print(f"Excluidos por caso incompleto: {audit['excluded_incomplete']/audit['private_households']:.3%}")
print('\nNo utilizables por variable, antes del filtro conjunto:')
print(pd.Series(audit['missing_by_variable']).to_string())
regional = pd.DataFrame(model['regions'])
regional['excluidos'] = regional.private_households - regional.complete_households
regional['pct_exclusion'] = regional.excluidos / regional.private_households * 100
print('\nCalidad por región:')
print(regional.to_string(index=False))
assert counts[:,0].sum() == model['sample_count']
assert np.array_equal(counts[:,0],counts[:,1:].sum(axis=1))


household_file_rows : 6,622,597
excluded_other_operations : 26,070
private_households : 6,596,527
complete_households : 6,581,257
excluded_incomplete : 15,270
dwelling_file_rows : 7,664,466
private_household_dwellings : 6,408,172
duplicate_households : 0
join_unmatched : 0
Excluidos por caso incompleto: 0.231%

No utilizables por variable, antes del filtro conjunto:
area                           0
p12_tenencia_viv            1096
tipologia_hogar                0
p6_fuente_agua              1373
indice_hacinamiento         1066
p15b_serv_compu            10785
p15d_serv_internet_fija     7571

Calidad por región:
 code              label  private_households  complete_households  excluidos  pct_exclusion
    1           Tarapacá              121511               121236        275       0.226317
    2        Antofagasta              213747               213044        703       0.328894
    3            Atacama              103057               102697        360       0.349321
    4      

## Análisis exploratorio de frecuencias y posible sesgo de exclusión

Las marginales previas al filtro permiten comparar cada variable entre quienes
la respondieron y quienes además respondieron las otras variables. Estas dos
proporciones tienen denominadores distintos. Una diferencia pequeña no descarta
sesgo en perfiles específicos, ni errores de cobertura o respuesta del censo.


In [4]:
rows = []
for item in model['items']:
    marginal = audit['marginals_before_complete_case_exclusion'][item['source_variable']]
    valid_n = sum(int(marginal.get(str(v),0)) for v in range(1,len(next(g[3] for g in study.GROUPS if g[0] == item['group']))+1))
    before = int(marginal.get(str(item['value']),0))
    after = study.item_count(counts[:,0].sum(axis=0),[item['code']])
    rows.append({'Categoría':item['label'],'Respuesta válida':before,'Casos completos':after,
                 'Pct respuesta válida':100*before/valid_n,'Pct casos completos':100*after/model['sample_count']})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('\nPerfiles no vacíos:',int(np.count_nonzero(counts[:,0].sum(axis=0))))
print('Los conteos ponderados conservan exactamente el número de hogares, sin expandir una matriz one-hot.')


                                   Categoría  Respuesta válida  Casos completos  Pct respuesta válida  Pct casos completos
                                      Urbana           5708503          5695447                86.538               86.540
                                       Rural            888024           885810                13.462               13.460
                               Propia pagada           3120238          3114211                47.309               47.319
                            Propia pagándose            910247           908130                13.801               13.799
                      Arrendada con contrato           1229328          1226646                18.639               18.638
                      Arrendada sin contrato            498871           497603                 7.564                7.561
               Cedida por trabajo o servicio             90697            90476                 1.375                1.375
                

## Experimento y criterios previos

El protocolo se escribió antes de ejecutar la minería. Se separan viviendas,
mediante hash determinista, en 70% descubrimiento, 15% selección y 15% prueba.
Los hogares de una misma vivienda permanecen juntos. Esta partición es interna
y transversal: no es una predicción del futuro ni validación con otra fuente.

Se compara Apriori con soporte/confianza mínimos 0,5%/40% y 2%/60%; el lift de
descubrimiento debe ser >=1,2. Para una regla de dos condiciones, la confianza
debe superar en al menos 5 puntos porcentuales a cada condición aislada.
La selección usa soporte >=0,3%, lift >=1,15, al menos 1.000 antecedentes,
una confianza no inferior al umbral de su configuración menos 5 puntos, y
variación absoluta de confianza <=5 puntos respecto al descubrimiento.

Primero se compara cobertura de categorías consecuentes; luego retención de
candidatas, menor longitud media del antecedente y mayor soporte mínimo. La
configuración y el orden quedan guardados antes de calcular la prueba.


In [5]:
print('Hogares por partición:',model['split_counts'])
print(pd.DataFrame(model['comparison']).to_string(index=False))
print('Configuración elegida:',model['selected_configuration'])
decision = json.loads((out / 'seleccion_cerrada.json').read_text())
assert decision['rule_ids'] == [r['id'] for r in model['rules']]
print('Ambas configuraciones cubren 4 categorías y retienen todas sus candidatas. La amplia gana por menor longitud media del antecedente (1,479 frente a 1,529).')
print('Este desempate es una decisión del protocolo, no evidencia de superioridad universal.')


Hogares por partición: {'discovery': 4606525, 'selection': 988197, 'test': 986535}
        name  min_support  min_confidence  candidate_rules  eligible_rules  consequent_categories  retention  mean_antecedent_size
      amplia        0.005             0.4               48              48                      4        1.0              1.479167
conservadora        0.020             0.6               17              17                      4        1.0              1.529412
Configuración elegida: amplia
Ambas configuraciones cubren 4 categorías y retienen todas sus candidatas. La amplia gana por menor longitud media del antecedente (1,479 frente a 1,529).
Este desempate es una decisión del protocolo, no evidencia de superioridad universal.


## Prueba interna y resultados descriptivos

Se calculan las métricas de prueba de todas las reglas elegidas sin filtrarlas
ni reordenarlas. Los soportes usan todos los hogares completos de cada partición;
la confianza usa los hogares que cumplen el antecedente y el lift compara con
la frecuencia basal del consecuente. No hay etiqueta objetivo para calcular
accuracy o F1. Tampoco se muestran intervalos de muestreo como si esta base fuera
una encuesta probabilística.


In [6]:
table = pd.read_csv(out / 'reglas.csv')
print(table[['rule_id','test_antecedent_count','test_joint_count','test_support','test_confidence','test_lift']].to_string(index=False))
summary = json.loads((out / 'resumen.json').read_text())
print('\nLift mínimo en prueba:',summary['test_min_lift'])
print('Máxima diferencia de confianza entre descubrimiento y prueba:',summary['test_max_confidence_drift'])
rule = next(r for r in model['rules'] if set(r['antecedent']) == {'area:2','tipologia:1'} and r['consequent'] == 'internet:2')
print('\nEjemplo: hogares rurales unipersonales → sin internet fija')
print(json.dumps(rule['population'],ensure_ascii=False,indent=2))
print('La ausencia de internet fija no informa por sí sola sobre internet móvil o satelital.')


                                    rule_id  test_antecedent_count  test_joint_count  test_support  test_confidence  test_lift
         area:2 + tipologia:1 -> internet:2                  29126             25801      0.026153         0.885841   2.493823
                       agua:4 -> internet:2                  14704             12889      0.013065         0.876564   2.467708
         agua:2 + tipologia:1 -> internet:2                   8242              6997      0.007093         0.848944   2.389952
          area:2 + tenencia:5 -> internet:2                   6858              5793      0.005872         0.844707   2.378023
                       agua:3 -> internet:2                  19923             16433      0.016657         0.824826   2.322053
                       area:2 -> internet:2                 133095            101901      0.103292         0.765626   2.155394
                   tenencia:8 -> internet:2                   8637              6283      0.006369         0.72

## Heterogeneidad territorial y contraste independiente

Una regla nacional puede tener menor lift, o incluso lift <1, en una región.
La comparación usa la frecuencia basal del propio territorio. Este contraste
descriptivo ayuda a detectar dependencia del contexto geográfico; no es un
ajuste causal ni una demostración automática de paradoja de Simpson.

El script `verificar_recuento_censo.py` vuelve a leer los CSV y realiza un merge
pandas independiente para la regla rural/unipersonal, sin utilizar los conteos
de perfiles. Verifica antecedentes y coincidencias en Chile y las 16 regiones.


In [7]:
local = pd.read_csv(out / 'reglas_por_region.csv')
reversed_lift = local[(local.antecedent_count >= 100) & (local.lift < 1)]
print('Reglas nacionales con lift <1 en al menos una región:',reversed_lift.rule_id.nunique())
print(reversed_lift[['rule_id','region','antecedent_count','confidence','baseline','lift']].to_string(index=False))
verification = json.loads((out / 'verificacion_independiente.json').read_text())
verified_rule = next(r for r in model['rules'] if r['id'] == verification['rule_id'])
for region, measured in verification['by_region'].items():
    expected = verified_rule['population'] if region == '0' else verified_rule['by_region'][region]
    assert measured['antecedent_count'] == expected['antecedent_count']
    assert measured['joint_count'] == expected['joint_count']
print('\nContraste independiente:',verification['method'])
print('Regla:',verification['rule_id'])
print('Chile:',verification['by_region']['0'])
print('Territorios comprobados:',len(verification['by_region']))


Reglas nacionales con lift <1 en al menos una región: 8
                                    rule_id  region  antecedent_count  confidence  baseline     lift
       agua:2 + tipologia:3 -> computador:2      11               226    0.473451  0.485035 0.976118
hacinamiento:2 + tenencia:4 -> computador:2      11               331    0.483384  0.485035 0.996596
     tenencia:1 + tipologia:3 -> internet:2       2              9516    0.239176  0.260951 0.916556
     tenencia:1 + tipologia:3 -> internet:2      15              4136    0.310687  0.312487 0.994239
                 tenencia:5 -> computador:2       1              2783    0.447718  0.488955 0.915663
                 tenencia:5 -> computador:2       3              2765    0.459313  0.505701 0.908269
                 tenencia:5 -> computador:2      12              2258    0.317095  0.400614 0.791522
                 tenencia:4 -> computador:2       9             23801    0.521197  0.550677 0.946465
                 tenencia:4 -> comp

## Aplicación, alcance y reproducibilidad

La aplicación consulta antecedentes exactos de una o dos condiciones, con región
opcional. Muestra reglas nacionales seleccionadas y métricas sobre todos los
casos completos del territorio. Las consultas regionales requieren al menos
100 antecedentes. Una combinación sin reglas no indica ausencia de asociación.

Puede servir para explorar patrones de acceso digital y preparar preguntas de
investigación o comparaciones territoriales. No decide dónde intervenir ni quién
debe recibir un beneficio; no incorpora ingresos, calidad de conexión o causas.
Los millones de filas aportan respaldo descriptivo, pero no garantizan utilidad
ni novedad. La revisión de pertinencia con el profesor sigue pendiente.

Reproducción: `descargar_censo.py`, `censo_2024.py`, `verificar_recuento_censo.py`,
`generar_notebook_censo.py`, y `python -m unittest discover -s tests -v`.
Se utiliza Python 3.12, pandas 2.2.3 y NumPy 2.3.5. Cada celda se ejecutó con el
intérprete y captura de stdout al generar este notebook; no mediante un kernel
Jupyter. Los originales permanecen intactos. Los datos y sus derivados llevan
atribución INE y licencia CC BY-SA 4.0; no se intenta reidentificar personas.
